In [ ]:
"""
Smart File Organizer
Automatically sorts files in a folder into categorized subfolders.
Also supports searching and listing Kaggle datasets from the command line.

Usage:
  python file_organizer.py <folder_path>          # organize a folder
  python file_organizer.py                         # organize current directory
  python file_organizer.py --dry-run <folder>      # simulate without moving
  python file_organizer.py --kaggle <query>        # search Kaggle datasets
  python file_organizer.py --kaggle-top            # list top trending datasets
  python file_organizer.py --kaggle-download <dataset_ref>  # download a dataset
"""

import os
import shutil
import sys
import json
import subprocess
from datetime import datetime

# ── Category definitions ──────────────────────────────────────────────────────
CATEGORIES = {
    "🖼️  Images":      [".jpg", ".jpeg", ".png", ".gif", ".bmp", ".svg", ".webp", ".ico"],
    "📄 Documents":    [".pdf", ".doc", ".docx", ".txt", ".ppt", ".pptx", ".xls", ".xlsx", ".odt"],
    "🎵 Audio":        [".mp3", ".wav", ".aac", ".flac", ".ogg", ".m4a"],
    "🎬 Videos":       [".mp4", ".mov", ".avi", ".mkv", ".wmv", ".flv", ".webm"],
    "📦  Archives":    [".zip", ".rar", ".tar", ".gz", ".7z", ".bz2"],
    "💻 Code":         [".py", ".js", ".ts", ".html", ".css", ".java", ".cpp", ".c", ".json", ".xml", ".sh"],
    "📊  Data":        [".csv", ".xlsx", ".sql", ".db", ".json", ".yaml", ".yml"],
    "🔤 Fonts":        [".ttf", ".otf", ".woff", ".woff2"],
    "⚙️  Executables": [".exe", ".msi", ".dmg", ".pkg", ".deb", ".apk"],
}


def get_category(extension: str) -> str:
    ext = extension.lower()
    for category, extensions in CATEGORIES.items():
        if ext in extensions:
            return category
    return "🗂️ Others"


# ── File organizer ────────────────────────────────────────────────────────────
def organize(folder: str, dry_run: bool = False) -> dict:
    """
    Organize files in folder. If dry_run=True, only simulate (no moves).
    Returns a summary dict.
    """
    if not os.path.isdir(folder):
        print(f"❌ '{folder}' is not a valid directory.")
        sys.exit(1)

    summary = {}
    moved = 0
    entries = [e for e in os.scandir(folder) if e.is_file()]

    if not entries:
        print("⚠️  No files found in the folder.")
        return summary

    print(f"\n{'🔍 DRY RUN — no files will be moved' if dry_run else '🗂️  Organizing files...'}")
    print(f"📁 Folder : {os.path.abspath(folder)}")
    print(f"📋 Files  : {len(entries)} found\n")
    print("-" * 55)

    for entry in entries:
        _, ext = os.path.splitext(entry.name)
        category = get_category(ext)
        dest_dir = os.path.join(folder, category)
        dest_path = os.path.join(dest_dir, entry.name)

        # Handle duplicate filenames
        if os.path.exists(dest_path):
            base, extension = os.path.splitext(entry.name)
            timestamp = datetime.now().strftime("%H%M%S")
            dest_path = os.path.join(dest_dir, f"{base}_{timestamp}{extension}")

        print(f"  {entry.name:<35} →  {category}")

        if not dry_run:
            os.makedirs(dest_dir, exist_ok=True)
            shutil.move(entry.path, dest_path)

        summary[category] = summary.get(category, 0) + 1
        moved += 1

    print("-" * 55)
    print(f"\n✅ {'Would move' if dry_run else 'Moved'} {moved} file(s)\n")
    print("📊 Summary:")
    for cat, count in sorted(summary.items()):
        print(f"   {cat:<28}  {count} file(s)")

    return summary


# ── Kaggle helpers ────────────────────────────────────────────────────────────
def _check_kaggle_installed() -> bool:
    """Return True if the kaggle CLI is available."""
    return shutil.which("kaggle") is not None


def _ensure_kaggle():
    """Install kaggle package if missing."""
    if not _check_kaggle_installed():
        print("📦 'kaggle' CLI not found. Installing via pip...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle", "--quiet"])
        print("✅ kaggle installed.\n")


def _kaggle_credentials_ok() -> bool:
    """Check whether ~/.kaggle/kaggle.json exists (minimal credential check)."""
    cred_path = os.path.expanduser("~/.kaggle/kaggle.json")
    return os.path.isfile(cred_path)


def _print_credential_help():
    print("\n⚠️  Kaggle credentials not found.")
    print("   To use Kaggle features you need an API key:")
    print("   1. Go to https://www.kaggle.com/account")
    print("   2. Click 'Create New Token' — this downloads kaggle.json")
    print("   3. Move it to ~/.kaggle/kaggle.json")
    print("   4. Run:  chmod 600 ~/.kaggle/kaggle.json\n")


def kaggle_search(query: str, max_results: int = 10):
    """Search Kaggle datasets by keyword and print a formatted table."""
    _ensure_kaggle()

    if not _kaggle_credentials_ok():
        _print_credential_help()
        sys.exit(1)

    print(f"\n🔍 Searching Kaggle datasets for: '{query}'\n")
    print("-" * 90)
    print(f"  {'#':<4} {'Dataset Ref':<40} {'Title':<35} {'Size':>6}")
    print("-" * 90)

    try:
        result = subprocess.run(
            ["kaggle", "datasets", "list", "--search", query,
             "--csv", "--max-size", "99999999999"],
            capture_output=True, text=True, check=True
        )
        lines = result.stdout.strip().splitlines()
        if len(lines) <= 1:
            print("  No datasets found for that query.")
            return

        # Skip CSV header row
        rows = lines[1:max_results + 1]
        for idx, row in enumerate(rows, start=1):
            parts = row.split(",")
            if len(parts) < 5:
                continue
            ref   = parts[0].strip()
            title = parts[1].strip()[:34]
            size  = parts[4].strip() if len(parts) > 4 else "?"
            print(f"  {idx:<4} {ref:<40} {title:<35} {size:>6}")

        print("-" * 90)
        print(f"\n💡 Tip: download any dataset with:")
        print(f"       python file_organizer.py --kaggle-download <dataset-ref>\n")

    except subprocess.CalledProcessError as e:
        print(f"❌ Kaggle CLI error:\n{e.stderr}")
        sys.exit(1)


def kaggle_top(max_results: int = 15):
    """List the most popular / trending Kaggle datasets."""
    _ensure_kaggle()

    if not _kaggle_credentials_ok():
        _print_credential_help()
        sys.exit(1)

    print(f"\n🏆 Top Trending Kaggle Datasets\n")
    print("-" * 90)
    print(f"  {'#':<4} {'Dataset Ref':<40} {'Title':<35} {'Size':>6}")
    print("-" * 90)

    try:
        result = subprocess.run(
            ["kaggle", "datasets", "list", "--sort-by", "hottest", "--csv"],
            capture_output=True, text=True, check=True
        )
        lines = result.stdout.strip().splitlines()
        if len(lines) <= 1:
            print("  No datasets returned.")
            return

        rows = lines[1:max_results + 1]
        for idx, row in enumerate(rows, start=1):
            parts = row.split(",")
            if len(parts) < 5:
                continue
            ref   = parts[0].strip()
            title = parts[1].strip()[:34]
            size  = parts[4].strip() if len(parts) > 4 else "?"
            print(f"  {idx:<4} {ref:<40} {title:<35} {size:>6}")

        print("-" * 90)
        print(f"\n💡 Download with:  python file_organizer.py --kaggle-download <dataset-ref>\n")

    except subprocess.CalledProcessError as e:
        print(f"❌ Kaggle CLI error:\n{e.stderr}")
        sys.exit(1)


def kaggle_download(dataset_ref: str, dest_folder: str = "."):
    """
    Download a Kaggle dataset by its ref (owner/dataset-name)
    into dest_folder, then organize the downloaded files automatically.
    """
    _ensure_kaggle()

    if not _kaggle_credentials_ok():
        _print_credential_help()
        sys.exit(1)

    dest = os.path.abspath(dest_folder)
    os.makedirs(dest, exist_ok=True)

    print(f"\n⬇️  Downloading dataset: {dataset_ref}")
    print(f"   → Destination: {dest}\n")

    try:
        subprocess.check_call([
            "kaggle", "datasets", "download",
            "--dataset", dataset_ref,
            "--path", dest,
            "--unzip"
        ])
        print(f"\n✅ Download complete!\n")
        print("🗂️  Now organizing downloaded files...")
        organize(dest)

    except subprocess.CalledProcessError as e:
        print(f"❌ Download failed: {e}")
        sys.exit(1)


# ── CLI entry point ───────────────────────────────────────────────────────────
def main():
    args = sys.argv[1:]

    # ── Kaggle search ─────────────────────────────────────────────────────────
    if "--kaggle" in args:
        idx = args.index("--kaggle")
        query_parts = args[idx + 1:]
        if not query_parts:
            print("❌ Please provide a search query, e.g.:  --kaggle titanic")
            sys.exit(1)
        kaggle_search(" ".join(query_parts))
        return

    # ── Kaggle top / trending ────────────────────────────────────────────────
    if "--kaggle-top" in args:
        kaggle_top()
        return

    # ── Kaggle download ──────────────────────────────────────────────────────
    if "--kaggle-download" in args:
        idx = args.index("--kaggle-download")
        remaining = [a for a in args[idx + 1:] if not a.startswith("--")]
        if not remaining:
            print("❌ Please provide a dataset ref, e.g.:  --kaggle-download zillow/zecon")
            sys.exit(1)
        dataset_ref = remaining[0]
        dest = remaining[1] if len(remaining) > 1 else "."
        kaggle_download(dataset_ref, dest)
        return

    # ── Standard file organizer ──────────────────────────────────────────────
    dry_run = "--dry-run" in args
    paths   = [a for a in args if not a.startswith("--")]
    folder  = paths[0] if paths else "."

    organize(folder, dry_run=dry_run)
    print("\nDone! ✅")


if __name__ == "__main__":
    main()